In [0]:
!pip install pyspark -q
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("S5Clase").getOrCreate()
print(f"✅ Spark {spark.version}")

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
✅ Spark 4.1.0


In [0]:
# ============================================================
# CELDA 1: Dataset de pedidos PidelitoApp Lima
# Ejecutar tal cual — no modificar
# ============================================================
import numpy as np
from pyspark.sql import functions as F
from pyspark.sql.types import *

np.random.seed(42)
N = 800

restaurantes = ["La Mar", "Tanta", "Astrid & Gastón", "El Hornero",
                "Bembos", "KFC Lima", "Pardos Chicken", "Sushi Pop",
                "Cevichería Isolina", "Pizza Hut Miraflores"]
distritos    = ["Miraflores","San Isidro","Barranco","Surco","San Borja",
                "Los Olivos","Ate","SJL","Callao","Villa El Salvador"]
estados      = ["entregado","entregado","entregado","cancelado","en_camino"]

pedidos = spark.createDataFrame([
    (f"PD{i:06d}",
     str(np.random.choice(restaurantes)),  # <-- CORRECCIÓN: str() añadido
     str(np.random.choice(distritos)),     # <-- CORRECCIÓN: str() añadido
     int(np.random.randint(7, 23)),
     float(round(np.random.uniform(18, 120), 2)),
     float(round(np.random.uniform(3, 12), 2)),
     int(np.random.randint(15, 65)),
     float(round(np.random.normal(4.2, 0.5), 1)),
     f"REP{np.random.randint(1,40):03d}",
     str(np.random.choice(estados, p=[.75,.08,.05,.10,.02])))
    for i in range(1, N+1)
], ["id_pedido", "restaurante", "distrito", "hora", "monto_soles", 
    "costo_delivery", "tiempo_entrega_min", "rating", "id_repartidor", "estado"])

pedidos.createOrReplaceTempView("pedidos")
print(f"✅ {pedidos.count()} pedidos cargados")
pedidos.show(3)

✅ 800 pedidos cargados
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
|id_pedido|   restaurante|  distrito|hora|monto_soles|costo_delivery|tiempo_entrega_min|rating|id_repartidor|   estado|
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
| PD000001|Pardos Chicken|     Surco|  19|      36.71|         10.02|                35|   4.1|       REP011|entregado|
| PD000002|        Bembos|     Surco|  14|      90.22|          3.19|                16|   4.1|       REP024|entregado|
| PD000003|      KFC Lima|San Isidro|  22|     119.21|          8.56|                36|   3.0|       REP028|cancelado|
+---------+--------------+----------+----+-----------+--------------+------------------+------+-------------+---------+
only showing top 3 rows


In [0]:
# ============================================================
# CELDA 2: Consulta 1 — Top 5 restaurantes por ingresos
# SQL: SELECT restaurante, COUNT, SUM(monto), AVG(rating)
#      WHERE estado = 'entregado'
#      GROUP BY restaurante
#      ORDER BY ingresos DESC LIMIT 5
# ============================================================

top_restaurantes = spark.sql("""
    SELECT 
        restaurante,
        COUNT(*)                        AS total_pedidos,
        ROUND(SUM(monto_soles), 2)      AS ingresos_totales,
        ROUND(AVG(rating), 2)           AS rating_prom,
        ROUND(AVG(tiempo_entrega_min),1)AS tiempo_entrega_prom
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY restaurante
    ORDER BY ingresos_totales DESC
    LIMIT 5
""")

print("🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES:")
top_restaurantes.show(truncate=False)

🍽️ TOP 5 RESTAURANTES — INGRESOS DEL MES:
+--------------------+-------------+----------------+-----------+-------------------+
|restaurante         |total_pedidos|ingresos_totales|rating_prom|tiempo_entrega_prom|
+--------------------+-------------+----------------+-----------+-------------------+
|Tanta               |86           |6280.6          |4.28       |37.4               |
|Pardos Chicken      |85           |5702.74         |4.28       |41.6               |
|La Mar              |83           |5643.05         |4.17       |39.1               |
|Pizza Hut Miraflores|81           |5475.86         |4.24       |40.9               |
|KFC Lima            |71           |4793.57         |4.13       |38.1               |
+--------------------+-------------+----------------+-----------+-------------------+



In [0]:
# =================================================================
# CELDA 3: ► TU TURNO - Consulta 2: Demanda por hora
# COMPLETA los ___ para responder:
# "¿A qué hora del día hay más pedidos y mayor ingreso?"
# =================================================================

demanda_hora = spark.sql("""
    SELECT 
        hora,
        COUNT(*)                                 AS total_pedidos,
        ROUND(SUM(monto_soles), 2)               AS ingresos_hora,
        ROUND(AVG(tiempo_entrega_min), 1)        AS tiempo_prom
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY hora
    ORDER BY total_pedidos DESC
""")

print("⏱ DEMANDA POR HORA DEL DÍA:")
demanda_hora.show(8)

⏱ DEMANDA POR HORA DEL DÍA:
+----+-------------+-------------+-----------+
|hora|total_pedidos|ingresos_hora|tiempo_prom|
+----+-------------+-------------+-----------+
|   7|           55|      3548.36|       40.6|
|  15|           51|      3303.32|       42.2|
|  18|           50|       3402.1|       36.7|
|  13|           50|       3527.5|       40.8|
|  10|           48|      3399.76|       39.3|
|  19|           47|      3228.09|       39.9|
|  11|           46|      3404.92|       38.4|
|   8|           44|      2756.09|       37.0|
+----+-------------+-------------+-----------+
only showing top 8 rows


In [0]:
# =================================================================
# CELDA 4: ► TU TURNO - Consulta 3: Repartidores eficientes
# COMPLETA el pipeline para encontrar repartidores con:
# - Más de 10 pedidos entregados
# - Tiempo promedio de entrega < 40 min
# - Rating > 4.0
# =================================================================

repartidores_eficientes = spark.sql("""
    SELECT 
        id_repartidor,
        COUNT(*)                             AS pedidos_entregados,
        ROUND(AVG(tiempo_entrega_min), 1)    AS tiempo_prom,
        ROUND(AVG(rating), 2)                AS rating_prom,
        ROUND(SUM(costo_delivery), 2)        AS ingresos_delivery
    FROM pedidos
    WHERE estado = 'entregado'
    GROUP BY id_repartidor
    HAVING COUNT(*) > 10
        AND AVG(tiempo_entrega_min) < 40
        AND AVG(rating) > 4.0
    ORDER BY rating_prom DESC
""")

print("⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo<40min, rating>4.0):")
repartidores_eficientes.show(10)
print(f"Total repartidores que cumplen los criterios: {repartidores_eficientes.count()}")

⚡ REPARTIDORES EFICIENTES (>10 pedidos, tiempo<40min, rating>4.0):
+-------------+------------------+-----------+-----------+-----------------+
|id_repartidor|pedidos_entregados|tiempo_prom|rating_prom|ingresos_delivery|
+-------------+------------------+-----------+-----------+-----------------+
|       REP037|                17|       39.6|       4.43|           123.73|
|       REP031|                14|       38.1|       4.39|            96.85|
|       REP022|                12|       39.8|       4.38|           107.65|
|       REP016|                14|       39.5|       4.36|            95.27|
|       REP004|                22|       38.4|        4.3|           162.79|
|       REP014|                16|       35.3|       4.29|           120.01|
|       REP029|                23|       34.0|       4.29|           163.63|
|       REP034|                17|       39.5|       4.27|            149.4|
|       REP015|                14|       37.8|       4.27|            86.35|
|       R

# ============================================================
# CELDA 5: Escribe tus respuestas como comentarios Python
# ============================================================

pregunta_1 = """
¿Cuándo hay más pedidos según tu consulta (hora del día)?
¿Coincide con lo que esperabas para Lima? ¿Por qué sí o no?

# Tu respuesta:
"""

pregunta_2 = """
Si la empresa quiere bonificar a los mejores repartidores,
¿cuál de los 3 criterios de la Consulta 3 es el más importante?
Justifica con un argumento de negocio.

# Tu respuesta:
"""

pregunta_3 = """
¿Por qué usamos Spark SQL y no pandas para este análisis?
Da UN argumento técnico específico.

# Tu respuesta:
"""

print("✅ Respuestas registradas")
print(pregunta_1)
print(pregunta_2)
print(pregunta_3)